<a href="https://colab.research.google.com/github/suleymagination/mechanistic-tomato-crop-model/blob/main/notebooks/extract_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Extraction and Preprocessing Module
----------------------------------------
Extracts environmental time-series and harvest yield metrics from the raw
Wageningen University & Research (WUR) 4th Autonomous Greenhouse Challenge
dataset (Compartment 3.06 - Reference Strategy).

Source Dataset:

Maree, Stef; Zhang, Pinglin; van Marrewijk, B. M. (Bart); H.F. (Feije) de Zwart; monique bijlaard et. al. (2025): 4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images . Version 1. 4TU.ResearchData. dataset. https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1

In [11]:
from pathlib import Path
import pandas as pd


def extract_greenhouse_climate_data(
    input_filepath: str = "https://raw.githubusercontent.com/suleymagination/mechanistic-tomato-crop-model/main/data/reference.csv",
    output_filepath: str = "../data/greenhouse_climate.csv",
) -> pd.DataFrame:
    """Extracts and standardizes the complete microclimate time-series parameters."""
    target_columns = {
        "time": "time",
        "compartment/air_temperature": "temp_c",
        "compartment/relative_humidity": "rh_pct",
        "compartment/par": "par_ppfd",
        "compartment/co2_concentration": "co2_ppm",
        "dwarf_tomato/plant_density": "plant_density",
    }

    df_raw = pd.read_csv(input_filepath)

    # Forward-fill plant_density (spacing changes are only logged at event timestamps)
    df_raw["dwarf_tomato/plant_density"] = df_raw["dwarf_tomato/plant_density"].ffill()

    # Extract required columns and drop missing sensor readings
    df_climate = df_raw[list(target_columns.keys())].dropna()
    df_climate = df_climate.rename(columns=target_columns)

    # Standardize timestamp parsing
    df_climate["time"] = pd.to_datetime(df_climate["time"], utc=True)

    # Export complete dataset (21,168 records spanning full cultivation cycle)
    Path(output_filepath).parent.mkdir(parents=True, exist_ok=True)
    df_climate.to_csv(output_filepath, index=False)

    return df_climate


def extract_harvest_summary_data(
    input_filepath: str = "https://raw.githubusercontent.com/suleymagination/mechanistic-tomato-crop-model/main/data/Harvest.xlsx",
    output_filepath: str = "../data/harvest_summary.csv",
) -> pd.DataFrame:
    """Parses and consolidates all individual plant harvest metrics across all harvest dates for Compartment 3.06."""

    harvest_sheets = [
        ("Destructive harvest 7-10-24", "2024-10-07", "destructive_1"),
        ("Destructive harvest 22-10-24", "2024-10-22", "destructive_2"),
        ("Destructive harvest 05-11-24", "2024-11-05", "destructive_3"),
        ("Final Harvest  3.06", "2024-11-15", "final_harvest"),
    ]

    all_harvest_records = []

    for sheet_name, harvest_date, harvest_type in harvest_sheets:
        df_sheet = pd.read_excel(input_filepath, sheet_name=sheet_name)

        # Normalize column names (strip whitespace)
        df_sheet.columns = [str(c).strip() for c in df_sheet.columns]

        if "plant" not in df_sheet.columns:
            continue

        df_sheet["plant"] = df_sheet["plant"].astype(str).str.strip()
        # Filter strictly for individual plant records (306-1, 306-2, ..., 306-50)
        df_306 = df_sheet[df_sheet["plant"].str.startswith("306-")].copy()

        for _, row in df_306.iterrows():
            total_fw = row.get("FW tot", row.get("FW", None))
            total_count = row.get("# tot", row.get("#tot", row.get("# tom", None)))

            green_count = row.get("# tom green", None)
            green_fw = row.get("fw green", row.get("FW green", None))

            coloured_count = row.get("#tom coloured", None)
            coloured_fw = row.get("Fw coloured", row.get("Fw tom coloured", None))

            red_count = row.get("#tom red", None)
            red_fw = row.get("FW red", row.get("FW tom red", None))

            all_harvest_records.append({
                "harvest_date": harvest_date,
                "harvest_type": harvest_type,
                "compartment": "306",
                "plant_id": row["plant"],
                "total_fw_g": total_fw,
                "total_fruit_count": total_count,
                "green_fruit_count": green_count,
                "green_fw_g": green_fw,
                "coloured_fruit_count": coloured_count,
                "coloured_fw_g": coloured_fw,
                "red_fruit_count": red_count,
                "red_fw_g": red_fw,
            })

    df_harvest_summary = pd.DataFrame(all_harvest_records)

    # Export complete dataset (68 plant records across all harvest dates)
    Path(output_filepath).parent.mkdir(parents=True, exist_ok=True)
    df_harvest_summary.to_csv(output_filepath, index=False)

    return df_harvest_summary


if __name__ == "__main__":
    climate_data_url = "https://raw.githubusercontent.com/suleymagination/mechanistic-tomato-crop-model/main/data/reference.csv"
    harvest_data_url = "https://raw.githubusercontent.com/suleymagination/mechanistic-tomato-crop-model/main/data/Harvest.xlsx"

    climate_df = extract_greenhouse_climate_data(input_filepath=climate_data_url)
    harvest_df = extract_harvest_summary_data(input_filepath=harvest_data_url)

    # Display outputs, dataset dimensions, and plant density distribution
    print(f"Climate Dataset Extracted: {climate_df.shape[0]} rows, {climate_df.shape[1]} columns")
    print("--- Climate Data Head ---")
    display(climate_df.head(5))

    print("\n--- Plant Density Distribution (5-minute interval counts) ---")
    display(climate_df["plant_density"].value_counts().reset_index(name="count"))

    print(f"\nHarvest Dataset Extracted: {harvest_df.shape[0]} rows, {harvest_df.shape[1]} columns")

    print("\n--- Harvest Summary Head ---")
    display(harvest_df.head(5))

    print("\n--- Harvest Summary Breakdown by Date ---")
    display(harvest_df.groupby(["harvest_date", "harvest_type"]).size().reset_index(name="plant_count"))

Climate Dataset Extracted: 21168 rows, 6 columns
--- Climate Data Head ---


,time,temp_c,rh_pct,par_ppfd,co2_ppm,plant_density
1,2024-09-02 22:05:00+00:00,22.8,91.4,0.0,438.0,56.0
2,2024-09-02 22:10:00+00:00,23.1,91.2,0.0,438.0,56.0
3,2024-09-02 22:15:00+00:00,23.2,91.2,0.0,440.0,56.0
4,2024-09-02 22:20:00+00:00,23.4,91.1,0.0,441.0,56.0
5,2024-09-02 22:25:00+00:00,23.2,91.1,0.0,444.0,56.0



--- Plant Density Distribution (5-minute interval counts) ---


,plant_density,count
0,20.0,11377
1,30.0,5184
2,42.0,2304
3,56.0,2303



Harvest Dataset Extracted: 68 rows, 12 columns

--- Harvest Summary Head ---


,harvest_date,harvest_type,compartment,plant_id,total_fw_g,total_fruit_count,green_fruit_count,green_fw_g,coloured_fruit_count,coloured_fw_g,red_fruit_count,red_fw_g
0,2024-10-07,destructive_1,306,306-1,77.92,35.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-07,destructive_1,306,306-2,71.60,36.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-07,destructive_1,306,306-3,54.31,30.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-07,destructive_1,306,306-4,76.03,39.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-07,destructive_1,306,306-5,45.44,29.0,NaN,NaN,NaN,NaN,NaN,NaN



--- Harvest Summary Breakdown by Date ---


,harvest_date,harvest_type,plant_count
0,2024-10-07,destructive_1,6
1,2024-10-22,destructive_2,6
2,2024-11-05,destructive_3,6
3,2024-11-15,final_harvest,50
